# Q-Shield: Ablation Study + Cross-Dataset Generalization

**Goal:** Build Table V (ablation) and Table IV (generalization) for the paper.

### Ablation variants (each trained from scratch)
| # | Variant | Expected effect |
|---|---------|----------------|
| A1 | Full v3 (ours) | baseline |
| A2 | No Siamese pretraining (random init head + fine-tune backbone) | big drop |
| A3 | BCE loss instead of focal | higher FNR |
| A4 | No frozen start (end-to-end from epoch 1) | minor drop |
| A5 | Smaller head (256→64→1 like v1) | minor drop |

### Cross-dataset splits
- **CV1:** Train CIC → Test Trad
- **CV2:** Train Trad → Test CIC
- **CV3:** Train combined → Test combined (main result, v3)

**Author:** Nicolas A. Llerena Silva (UTEC)

In [ ]:
# ============================================================
# 0. SETUP (shared with notebooks 06, 07)
# ============================================================
import sys, os, glob
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/QShield'
else:
    BASE = '.'

!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from PIL import Image
import pickle, zipfile, random, time, copy, json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================
WORK = '/content/qshield_data'
os.makedirs(WORK, exist_ok=True)

trad_dir = os.path.join(WORK, 'trad')
if not os.path.exists(os.path.join(trad_dir, 'qr_codes_29.pickle')):
    os.makedirs(trad_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(BASE, 'QuishingDataset.zip')) as z:
        z.extractall(trad_dir)
with open(os.path.join(trad_dir, 'qr_codes_29.pickle'), 'rb') as f:
    trad_qr = pickle.load(f)
with open(os.path.join(trad_dir, 'qr_codes_29_labels.pickle'), 'rb') as f:
    trad_labels = pickle.load(f)

cic_b_dir = os.path.join(WORK, 'cic_benign')
cic_m_dir = os.path.join(WORK, 'cic_malicious')
if not os.path.exists(cic_b_dir) or len(os.listdir(cic_b_dir)) == 0:
    os.makedirs(cic_b_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(BASE, 'QR_benign_430K.zip')) as z:
        z.extractall(cic_b_dir)
if not os.path.exists(cic_m_dir) or len(os.listdir(cic_m_dir)) == 0:
    os.makedirs(cic_m_dir, exist_ok=True)
    with zipfile.ZipFile(os.path.join(BASE, 'QR_malicious_576K.zip')) as z:
        z.extractall(cic_m_dir)

cic_b_files = sorted(glob.glob(os.path.join(cic_b_dir, '**', '*.png'), recursive=True))
cic_m_files = sorted(glob.glob(os.path.join(cic_m_dir, '**', '*.png'), recursive=True))
random.seed(SEED)
cic_b_files = random.sample(cic_b_files, min(30000, len(cic_b_files)))  # smaller for ablation speed
cic_m_files = random.sample(cic_m_files, min(30000, len(cic_m_files)))
print(f'Loaded: Trad={len(trad_qr):,}, CIC_b={len(cic_b_files):,}, CIC_m={len(cic_m_files):,}')

In [ ]:
# ============================================================
# 2. MODEL + DATASETS (identical to notebook 06)
# ============================================================

class MobileNetV2Embedding(nn.Module):
    def __init__(self, emb_dim=128, pretrained=True, dropout=0.35):
        super().__init__()
        mn = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
        orig = mn.features[0][0]
        self.features = mn.features
        self.features[0][0] = nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False)
        if pretrained:
            with torch.no_grad():
                self.features[0][0].weight = nn.Parameter(orig.weight.mean(dim=1, keepdim=True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Dropout(dropout), nn.Linear(512, emb_dim),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return F.normalize(self.projection(x), p=2, dim=1)

class SiameseQRNet(nn.Module):
    def __init__(self, emb_dim=128, pretrained=True, dropout=0.35):
        super().__init__()
        self.backbone = MobileNetV2Embedding(emb_dim, pretrained, dropout)
    def forward_one(self, x): return self.backbone(x)
    def forward(self, x1, x2): return self.backbone(x1), self.backbone(x2)

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.5):
        super().__init__(); self.margin = margin
    def forward(self, e1, e2, y):
        d = F.pairwise_distance(e1, e2)
        return ((1-y)*0.5*d.pow(2) + y*0.5*F.relu(self.margin-d).pow(2)).mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__(); self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        pt = p * targets + (1-p) * (1-targets)
        at = self.alpha * targets + (1-self.alpha) * (1-targets)
        return (at * (1-pt).pow(self.gamma) * bce).mean()

class QRClassifier(nn.Module):
    def __init__(self, backbone, emb_dim=128, head_size='large'):
        super().__init__()
        self.backbone = backbone
        if head_size == 'large':
            self.head = nn.Sequential(
                nn.Linear(emb_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.4),
                nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.3),
                nn.Linear(128, 32), nn.ReLU(True), nn.Dropout(0.2),
                nn.Linear(32, 1),
            )
        else:  # small
            self.head = nn.Sequential(
                nn.Linear(emb_dim, 256), nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.3),
                nn.Linear(256, 64), nn.ReLU(True), nn.Dropout(0.2),
                nn.Linear(64, 1),
            )
    def set_backbone_grad(self, rg):
        for p in self.backbone.parameters(): p.requires_grad = rg
    def forward(self, x):
        return self.head(self.backbone(x))

train_aug = T.Compose([T.RandomHorizontalFlip(p=0.5)])

class TradPairDataset(Dataset):
    def __init__(self, qr, labels, n, augment=False):
        self.qr = qr.astype(np.float32); self.labels = np.array(labels); self.n = n
        self.idx = {0: np.where(self.labels==0)[0], 1: np.where(self.labels==1)[0]}
        self.aug = train_aug if augment else None
    def __len__(self): return self.n
    def _tensor(self, i):
        t = torch.from_numpy(self.qr[i]).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=(224,224), mode='bilinear', align_corners=False).squeeze(0)
        return self.aug(t) if self.aug else t
    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0,1]); i1 = random.choice(self.idx[c])
        c2 = c if same else 1-c; i2 = random.choice(self.idx[c2])
        return self._tensor(i1), self._tensor(i2), torch.tensor(0.0 if same else 1.0)

class CICPairDataset(Dataset):
    def __init__(self, b, m, n, augment=False):
        self.f = {0: b, 1: m}; self.n = n
        self.aug = train_aug if augment else None
    def __len__(self): return self.n
    def _load(self, c, i):
        img = Image.open(self.f[c][i]).convert('L').resize((224,224))
        t = torch.from_numpy(np.array(img, dtype=np.float32)/255.0).unsqueeze(0)
        return self.aug(t) if self.aug else t
    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0,1]); i1 = random.randint(0, len(self.f[c])-1)
        c2 = c if same else 1-c; i2 = random.randint(0, len(self.f[c2])-1)
        return self._load(c,i1), self._load(c2,i2), torch.tensor(0.0 if same else 1.0)

class ClassifyDataset(Dataset):
    def __init__(self, trad_qr=None, trad_labels=None, cic_b=None, cic_m=None, augment=False):
        self.items = []
        if trad_qr is not None:
            for i in range(len(trad_qr)): self.items.append(('trad', i, int(trad_labels[i])))
            self.trad_qr = trad_qr.astype(np.float32)
        if cic_b is not None:
            for i,_ in enumerate(cic_b): self.items.append(('cic_b', i, 0))
            for i,_ in enumerate(cic_m): self.items.append(('cic_m', i, 1))
            self.cic_b = cic_b; self.cic_m = cic_m
        self.aug = train_aug if augment else None
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        src, i, lbl = self.items[idx]
        if src == 'trad':
            t = torch.from_numpy(self.trad_qr[i]).unsqueeze(0).unsqueeze(0)
            t = F.interpolate(t, size=(224,224), mode='bilinear', align_corners=False).squeeze(0)
        else:
            files = self.cic_b if src == 'cic_b' else self.cic_m
            img = Image.open(files[i]).convert('L').resize((224,224))
            t = torch.from_numpy(np.array(img, dtype=np.float32)/255.0).unsqueeze(0)
        if self.aug: t = self.aug(t)
        return t, torch.tensor(float(lbl))

print('All components defined.')

In [ ]:
# ============================================================
# 3. COMMON SPLITS
# ============================================================
idx_tr, idx_val = train_test_split(np.arange(len(trad_labels)), test_size=0.2,
                                    stratify=trad_labels, random_state=SEED)
qr_tr, lab_tr = trad_qr[idx_tr], trad_labels[idx_tr]
qr_val, lab_val = trad_qr[idx_val], trad_labels[idx_val]

sb = int(len(cic_b_files)*0.8); sm = int(len(cic_m_files)*0.8)
cic_b_tr, cic_b_val = cic_b_files[:sb], cic_b_files[sb:]
cic_m_tr, cic_m_val = cic_m_files[:sm], cic_m_files[sm:]

BATCH = 128
NUM_WORKERS = 4 if IN_COLAB else 0
print(f'Trad: {len(qr_tr)} train / {len(qr_val)} val')
print(f'CIC:  {sb*2} train / {(len(cic_b_files)-sb)*2} val')

---
## 4. TRAINING HELPER (compact version for ablation)

In [ ]:
# ============================================================
# 4. QUICK TRAIN HELPER (reduced epochs for ablation)
# ============================================================

def train_siamese(variant_name, epochs=10, pairs=20000,
                   pretrain=True, margin=1.5, dropout=0.35,
                   train_pair_ds=None, val_pair_ds=None):
    """Quick Siamese training for ablation study."""
    print(f'\n=== Training Siamese: {variant_name} ===')
    model = SiameseQRNet(128, pretrained=pretrain, dropout=dropout).to(device)
    criterion = ContrastiveLoss(margin=margin)
    opt = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=2e-4)

    tl = DataLoader(train_pair_ds, batch_size=BATCH, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True)
    vl = DataLoader(val_pair_ds, batch_size=BATCH, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    best_vl = float('inf'); best_st = None

    for ep in range(1, epochs+1):
        model.train()
        for x1, x2, y in tl:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            opt.zero_grad()
            e1, e2 = model(x1, x2)
            criterion(e1, e2, y).backward()
            opt.step()
        model.eval()
        v = 0; n = 0
        with torch.no_grad():
            for x1, x2, y in vl:
                x1, x2, y = x1.to(device), x2.to(device), y.to(device)
                e1, e2 = model(x1, x2)
                v += criterion(e1, e2, y).item()*x1.size(0); n += y.size(0)
        va = v/n
        if va < best_vl:
            best_vl = va; best_st = copy.deepcopy(model.state_dict())
        print(f'  Ep {ep}: val_loss={va:.4f}')
    model.load_state_dict(best_st)
    return model

def train_classifier(backbone, variant_name, epochs=10, use_focal=True,
                      frozen_ep=3, head_size='large',
                      tr_cls_ds=None, val_cls_ds=None):
    """Phase 2 training for ablation."""
    print(f'=== Training Classifier: {variant_name} ===')
    clf = QRClassifier(backbone, emb_dim=128, head_size=head_size).to(device)
    loss_fn = FocalLoss(alpha=0.5, gamma=2.0) if use_focal else nn.BCEWithLogitsLoss()

    tl = DataLoader(tr_cls_ds, batch_size=256, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True)
    vl = DataLoader(val_cls_ds, batch_size=512, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)

    if frozen_ep > 0:
        clf.set_backbone_grad(False)
        opt = optim.AdamW([p for p in clf.parameters() if p.requires_grad],
                          lr=5e-4, weight_decay=1e-4)
    else:
        opt = optim.AdamW(clf.parameters(), lr=1e-4, weight_decay=2e-4)

    best_auc = 0; best_st = None
    for ep in range(1, epochs+1):
        if ep == frozen_ep + 1 and frozen_ep > 0:
            clf.set_backbone_grad(True)
            opt = optim.AdamW(clf.parameters(), lr=1e-4, weight_decay=2e-4)
        clf.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device).unsqueeze(1)
            opt.zero_grad()
            loss_fn(clf(x), y).backward()
            opt.step()
        clf.eval()
        probs, true = [], []
        with torch.no_grad():
            for x, y in vl:
                logits = clf(x.to(device))
                probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
                true.extend(y.numpy().flatten())
        auc = roc_auc_score(true, probs)
        if auc > best_auc:
            best_auc = auc; best_st = copy.deepcopy(clf.state_dict())
        print(f'  Ep {ep}: AUC={auc:.4f}')
    clf.load_state_dict(best_st)
    print(f'  Best AUC: {best_auc:.4f}')
    return clf, best_auc

def evaluate(clf, loader):
    clf.eval()
    probs, true = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = clf(x.to(device))
            probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            true.extend(y.numpy().flatten())
    probs, true = np.array(probs), np.array(true)
    preds = (probs >= 0.5).astype(int)
    auc = roc_auc_score(true, probs)
    f1 = f1_score(true, preds)
    fnr = 1 - recall_score(true, preds)
    return auc, f1, fnr

---
## 5. ABLATION STUDY

In [ ]:
# ============================================================
# 5. ABLATION VARIANTS
# ============================================================

# Combined datasets
train_pair_combined = ConcatDataset([
    TradPairDataset(qr_tr, lab_tr, 20000, augment=True),
    CICPairDataset(cic_b_tr, cic_m_tr, 20000, augment=True),
])
val_pair_combined = ConcatDataset([
    TradPairDataset(qr_val, lab_val, 4000, augment=False),
    CICPairDataset(cic_b_val, cic_m_val, 4000, augment=False),
])
tr_cls_combined = ClassifyDataset(qr_tr, lab_tr, cic_b_tr, cic_m_tr, augment=True)
val_cls_combined = ClassifyDataset(qr_val, lab_val, cic_b_val, cic_m_val, augment=False)
val_loader_combined = DataLoader(val_cls_combined, batch_size=512, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True)

ablation_results = []

# A1: Full v3 (reference — use already-trained checkpoint, don't retrain)
print('A1: v3 full (from checkpoint)')
backbone_a1 = MobileNetV2Embedding(128, pretrained=False, dropout=0.35)
clf_a1 = QRClassifier(backbone_a1, emb_dim=128, head_size='large').to(device)
clf_a1.load_state_dict(torch.load(os.path.join(BASE, 'classifier_v3_phase2.pth'), map_location=device))
auc, f1, fnr = evaluate(clf_a1, val_loader_combined)
ablation_results.append(('A1: Full v3 (ours)', auc, f1, fnr))
print(f'  AUC={auc:.4f} F1={f1:.4f} FNR={fnr:.4f}')

# A2: No Siamese pretraining (skip Phase 1, random init)
print('\nA2: No Siamese pretraining')
backbone_no_pre = MobileNetV2Embedding(128, pretrained=True, dropout=0.35)
clf_a2, _ = train_classifier(backbone_no_pre, 'A2', epochs=8, use_focal=True,
                              frozen_ep=0, head_size='large',
                              tr_cls_ds=tr_cls_combined, val_cls_ds=val_cls_combined)
auc, f1, fnr = evaluate(clf_a2, val_loader_combined)
ablation_results.append(('A2: No Siamese pretraining', auc, f1, fnr))

# A3: Siamese + BCE (no focal)
print('\nA3: BCE instead of focal')
sia_a3 = train_siamese('A3', epochs=8, pretrain=True,
                        train_pair_ds=train_pair_combined, val_pair_ds=val_pair_combined)
clf_a3, _ = train_classifier(sia_a3.backbone, 'A3', epochs=8, use_focal=False,
                              frozen_ep=3, head_size='large',
                              tr_cls_ds=tr_cls_combined, val_cls_ds=val_cls_combined)
auc, f1, fnr = evaluate(clf_a3, val_loader_combined)
ablation_results.append(('A3: BCE loss (no focal)', auc, f1, fnr))

# A4: No frozen start
print('\nA4: End-to-end from epoch 1 (no frozen start)')
sia_a4 = train_siamese('A4', epochs=8, pretrain=True,
                        train_pair_ds=train_pair_combined, val_pair_ds=val_pair_combined)
clf_a4, _ = train_classifier(sia_a4.backbone, 'A4', epochs=8, use_focal=True,
                              frozen_ep=0, head_size='large',
                              tr_cls_ds=tr_cls_combined, val_cls_ds=val_cls_combined)
auc, f1, fnr = evaluate(clf_a4, val_loader_combined)
ablation_results.append(('A4: No frozen start', auc, f1, fnr))

# A5: Small head (v1 style)
print('\nA5: Small classifier head (256→64→1)')
sia_a5 = train_siamese('A5', epochs=8, pretrain=True,
                        train_pair_ds=train_pair_combined, val_pair_ds=val_pair_combined)
clf_a5, _ = train_classifier(sia_a5.backbone, 'A5', epochs=8, use_focal=True,
                              frozen_ep=3, head_size='small',
                              tr_cls_ds=tr_cls_combined, val_cls_ds=val_cls_combined)
auc, f1, fnr = evaluate(clf_a5, val_loader_combined)
ablation_results.append(('A5: Small head', auc, f1, fnr))

# Print table
print('\n' + '='*70)
print(' ABLATION STUDY RESULTS')
print('='*70)
df_ab = pd.DataFrame(ablation_results, columns=['Variant', 'AUC', 'F1', 'FNR'])
print(df_ab.to_string(index=False, float_format='%.4f'))

df_ab.to_csv(os.path.join(BASE, 'ablation_results.csv'), index=False)

---
## 6. CROSS-DATASET GENERALIZATION

In [ ]:
# ============================================================
# 6. CV1: Train CIC only → Test Trad only
# ============================================================

cross_results = []

# CV1 setup
print('CV1: Train CIC → Test Trad')
train_pair_cic = CICPairDataset(cic_b_tr, cic_m_tr, 40000, augment=True)
val_pair_cic = CICPairDataset(cic_b_val, cic_m_val, 6000, augment=False)
tr_cls_cic = ClassifyDataset(cic_b=cic_b_tr, cic_m=cic_m_tr, augment=True)
val_cls_cic = ClassifyDataset(cic_b=cic_b_val, cic_m=cic_m_val, augment=False)
val_cls_trad_only = ClassifyDataset(qr_val, lab_val, augment=False)
val_loader_trad_only = DataLoader(val_cls_trad_only, batch_size=512, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

sia_cv1 = train_siamese('CV1_CIC', epochs=8, pretrain=True,
                         train_pair_ds=train_pair_cic, val_pair_ds=val_pair_cic)
clf_cv1, _ = train_classifier(sia_cv1.backbone, 'CV1', epochs=8, use_focal=True,
                               frozen_ep=3, head_size='large',
                               tr_cls_ds=tr_cls_cic, val_cls_ds=val_cls_cic)
auc, f1, fnr = evaluate(clf_cv1, val_loader_trad_only)
cross_results.append(('CV1: Train CIC → Test Trad', auc, f1, fnr))
print(f'CV1 result on Trad: AUC={auc:.4f}')

# ============================================================
# CV2: Train Trad → Test CIC
# ============================================================
print('\nCV2: Train Trad → Test CIC')
train_pair_trad = TradPairDataset(qr_tr, lab_tr, 40000, augment=True)
val_pair_trad = TradPairDataset(qr_val, lab_val, 6000, augment=False)
tr_cls_trad_only = ClassifyDataset(qr_tr, lab_tr, augment=True)
val_cls_trad_only2 = ClassifyDataset(qr_val, lab_val, augment=False)
val_cls_cic_only = ClassifyDataset(cic_b=cic_b_val, cic_m=cic_m_val, augment=False)
val_loader_cic_only = DataLoader(val_cls_cic_only, batch_size=512, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True)

sia_cv2 = train_siamese('CV2_Trad', epochs=8, pretrain=True,
                         train_pair_ds=train_pair_trad, val_pair_ds=val_pair_trad)
clf_cv2, _ = train_classifier(sia_cv2.backbone, 'CV2', epochs=8, use_focal=True,
                               frozen_ep=3, head_size='large',
                               tr_cls_ds=tr_cls_trad_only, val_cls_ds=val_cls_trad_only2)
auc, f1, fnr = evaluate(clf_cv2, val_loader_cic_only)
cross_results.append(('CV2: Train Trad → Test CIC', auc, f1, fnr))
print(f'CV2 result on CIC: AUC={auc:.4f}')

# CV3: Combined (reference — already have this from v3)
print('\nCV3: Train Combined → Test Combined (v3 reference)')
cross_results.append(('CV3: Combined → Combined (v3)', 0.8962, 0.8205, 0.2019))

print('\n' + '='*70)
print(' CROSS-DATASET GENERALIZATION')
print('='*70)
df_cross = pd.DataFrame(cross_results, columns=['Setup', 'AUC', 'F1', 'FNR'])
print(df_cross.to_string(index=False, float_format='%.4f'))
df_cross.to_csv(os.path.join(BASE, 'crossdataset_results.csv'), index=False)

In [ ]:
# ============================================================
# 7. FINAL TABLES (for paper)
# ============================================================
print('\n' + '='*70)
print(' PAPER TABLE V — ABLATION STUDY')
print('='*70)
print(df_ab.to_string(index=False, float_format='%.4f'))

print('\n' + '='*70)
print(' PAPER TABLE IV — CROSS-DATASET GENERALIZATION')
print('='*70)
print(df_cross.to_string(index=False, float_format='%.4f'))

# Save JSON summary
summary = {
    'ablation': df_ab.to_dict(orient='records'),
    'cross_dataset': df_cross.to_dict(orient='records'),
}
with open(os.path.join(BASE, 'ablation_crossdataset_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nAll tables saved to {BASE}/')